# Banking77 model exploration

Explore how a fine-tuned Transformers model tokenizes banking support messages and produces intent logits.

## 1. Environment

Record the library versions and select the available inference device.

In [ ]:
import torch
import transformers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Setup

Load the tokenizer and sequence-classification model from the same checkpoint.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "mrm8488/distilroberta-finetuned-banking77"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID)
model.to(device)
model.eval()

print("Model:", MODEL_ID)
print("Device:", model.device)

## 3. Tokenizer

Inspect the tokens, token IDs and attention mask generated for one message.

In [ ]:
text = "I forgot my PIN!"
encoded = tokenizer(text, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

print("Tokens:", tokens)
for name, tensor in encoded.items():
    print(name, tuple(tensor.shape), tensor)

## 4. Model configuration

Inspect only the configuration values needed to understand this classifier.

In [ ]:
labels_to_inspect = ["passcode_forgotten", "pin_blocked", "change_pin"]

print("Model class:", type(model).__name__)
print("Labels:", model.config.num_labels)
print("Hidden layers:", model.config.num_hidden_layers)
print("Hidden size:", model.config.hidden_size)
for label in labels_to_inspect:
    print(label, "->", model.config.label2id[label])

## 5. Raw inference

Tokenize a batch, move it to the model device and inspect the raw logits.

In [ ]:
texts = [
    "I forgot my PIN",
    "Can I pay with Apple Pay?",
]

batch = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model(**batch)

logits = outputs.logits
print("Batch shape:", tuple(batch["input_ids"].shape))
print("Logits shape:", tuple(logits.shape))
print("First five logits:\n", logits[:, :5].cpu())

## 6. Scores and top-k

Convert the logits with softmax, select the highest scores and map their indices.

In [ ]:
scores = torch.softmax(logits.cpu(), dim=-1)

print("Score shape:", tuple(scores.shape))
print("Sum of each line:", torch.sum(scores, dim=-1))

In [ ]:
top_results = torch.topk(
    scores,
    k=3,
    dim=-1,
)

print("Top_k values:\n", top_results.values, "\n\tShape:", tuple(top_results.values.shape))
print("Top_k indices:\n", top_results.indices, "\n\tShape:", tuple(top_results.indices.shape))


In [ ]:
for text, indices, values in zip(
    texts,
    top_results.indices,
    top_results.values,
    strict=True,
):
    print(text)
    for index, score in zip(indices, values, strict=True):
        label = model.config.id2label[index.item()]
        print(f"  {label}: {score.item():.4f}")

## 7. Pipeline comparison

Compare the results with a Transformers text-classification pipeline.

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

pipeline_results = classifier(
    texts,
    top_k=3,
    truncation=True,
)

print(pipeline_results)